In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np


In [ ]:
import autosklearn
X_train=None
X_val=None
y_train=None
y_val=None
train=pd.read_csv("training_data_clean.csv")
X=train.drop(["ID_code",'target'],axis=1)
y=train["target"]
X_train,X_val,y_train,y_val = train_test_split(X,y, stratify=y,test_size=0.33, random_state=42)
#define the model
automl = autosklearn.classification.AutoSklearnClassifier()
#train the model
automl.fit(X_train, y_train )
#predict
y_pred=automl.predict_proba(X_val)
# score
score=roc_auc_score(y_val,y_pred[:,1])
print(score)
# show all models
show_modes_str=automl.show_models()
sprint_statistics_str = automl.sprint_statistics()

In [9]:
# SETUP

TRAINING_DATA = "~/Downloads/training_data_clean.csv"


# def prep_data():
df = pd.read_csv(TRAINING_DATA)

# rename
df.columns = [
    'id',
    'best_tasks_free',
    'acad_tasks_rating',
    'best_tasks_select',
    'subopt_freq_rating',
    'subopt_tasks_select',
    'subopt_tasks_free',
    'evidence_freq_rating',
    'verify_freq_rating',
    'verify_method_free',
    'target'
    ]

for task in df['best_tasks_select'].unique():
    print(task)

# prep_data()


Math computations,Data processing or analysis,Explaining complex concepts simply,Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés)
Writing or debugging code
Math computations,Converting content between formats (e.g., LaTeX)
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
Math computations,Writing or debugging code,Data processing or analysis,Converting content between formats (e.g., LaTeX)
Brainstorming or generating creative ideas
Writing or debugging code,Data processing or analysis,Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
nan
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Brainstormi

In [3]:
# def split_data(df):
students = df['id'].unique()
train_idx, test_idx = train_test_split(
    students, test_size=0.3, random_state=22
)

train_df = df[df['id'].isin(train_idx)]
test_df = df[df['id'].isin(test_idx)]

train_groups = train_df['id'].values
test_groups = test_df['id'].values

x_train = train_df.drop(columns=['target', 'id'])
y_train = train_df['target']

x_test = test_df.drop(columns=['target', 'id'])
y_test = test_df['target']

    # return x_train, x_test, y_train, y_test, train_groups, test_groups

In [ ]:
# EXPLORE DATA

df.isnull().sum()

id                       0
best_tasks_free          2
acad_tasks_rating        0
best_tasks_select       15
subopt_freq_rating       1
subopt_tasks_select     22
subopt_tasks_free       11
evidence_freq_rating    62
verify_freq_rating       4
verify_method_free      10
target                   0
dtype: int64

In [7]:
# PREPROCESSING 
# """These are EQUIVALENT:

# Option 1: make_pipeline (shorter)
# text_pipeline = make_pipeline(
#     SimpleImputer(strategy="constant", fill_value=""),
#     TfidfVectorizer(max_features=2000)
# )

# Option 2: Pipeline (explicit)
# text_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy="constant", fill_value="")),
#     ('tfidf', TfidfVectorizer(max_features=2000))
# ])"""

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder
from typing import Dict, List
import unicodedata


text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

# custom function makecorpus just to keep consistency in pipeline
class MakeCorpus(BaseEstimator, TransformerMixin):
    # required by TansformerMixin -ignore
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        # X is DataFrame with text columns
        return X.agg(' '.join, axis=1).tolist()

def preprocess_text():
    # TFIDF manually (classical)
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        # hardcoded stop_words param for now
        # TODO: try with max instead
        ('tfidf', TfidfVectorizer(stop_words='english'))
    ])

def preprocess_ord():

    return make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=ord_categories)

    )

def _normalize(s: str) -> str:
    # normalize unicode, collapse spaces, strip
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()

def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts

class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using a safe split and a fixed
    set of canonical choices. Clone-safe: __init__ does not modify params.
    If the original cell is NaN -> all dummies for that column are NaN.
    """
    def __init__(self, classes: List[str]):
        self.classes = classes  # DO NOT MODIFY HERE (clone-safe)

    # internal helpers use fitted attributes
    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        # keep only exact matches (normalized)
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        # create normalized, fitted classes (separate from init param)
        self.classes_ = [_normalize(c) for c in self.classes]

        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])  # fit on fixed classes only
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan  # preserve missingness for KNN later
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)

def preprocess_cat():
    return make_pipeline(MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices),
                                 SimpleImputer(strategy="constant", fill_value=0))

def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("text", preprocess_text(), text_cols), # pass in just the names of the columns for now, df specified later in full pipeline
        ("ord", preprocess_ord(), ord_cols),
         ("cat", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)


In [10]:

import numpy as np
import pandas as pd
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score, train_test_split
import optuna
from optuna.samplers import TPESampler
import joblib
import warnings
warnings.filterwarnings("ignore")
import lightgbm as lgb

# -------------------------
# USER CONFIG (edit here if needed)
# -------------------------
EMBEDDING_MODEL = "all-mpnet-base-v2"   # high-quality model for max accuracy
BATCH_SIZE = 64
RANDOM_STATE = 22
CV_SPLITS = 5
OPTUNA_TRIALS = 60   # increase if you have time
# -------------------------

# Example column lists — replace if your dataset differs
TEXT_COLS = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
]
ORD_COLS = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
]
CAT_COLS = [
    'best_tasks_select',
    'subopt_tasks_select',
]

# -------------------------
# Helper: split by unique ids -> grouped train/test
# -------------------------
def split_by_id(df: pd.DataFrame, id_col: str = "id", test_size: float = 0.3, random_state: int = RANDOM_STATE):
    students = df[id_col].unique()
    train_ids, test_ids = train_test_split(students, test_size=test_size, random_state=random_state)
    train_df = df[df[id_col].isin(train_ids)].reset_index(drop=True)
    test_df = df[df[id_col].isin(test_ids)].reset_index(drop=True)
    return train_df, test_df

# -------------------------
# Precompute per-column embeddings (returns DataFrame of emb_* columns)
# -------------------------
def precompute_embeddings(df: pd.DataFrame, text_cols: List[str], model_name: str = EMBEDDING_MODEL, batch_size: int = BATCH_SIZE):
    model = SentenceTransformer(model_name)
    emb_frames = []
    for col in text_cols:
        texts = df[col].fillna("").astype(str).tolist()
        print(f"Encoding column '{col}' ({len(texts)} rows) with {model_name} ...")
        emb = model.encode(texts, show_progress_bar=True, batch_size=batch_size)
        emb = np.asarray(emb)
        emb_cols = [f"emb__{col}__{i}" for i in range(emb.shape[1])]
        emb_df = pd.DataFrame(emb, columns=emb_cols, index=df.index)
        emb_frames.append(emb_df)
    emb_concat = pd.concat(emb_frames, axis=1)
    return emb_concat

# -------------------------
# Build feature matrix: per-column PCA then concat (preserves per-field structure)
# -------------------------
def build_embedding_features(emb_df: pd.DataFrame, n_components_per_col: Dict[str,int]=None, scale=True):
    # emb_df has columns like emb__{col}__{i}
    cols_by_field = {}
    for c in emb_df.columns:
        # column format: emb__{field}__{i}
        parts = c.split("__")
        if len(parts) >= 3:
            field = parts[1]
            cols_by_field.setdefault(field, []).append(c)
    out_frames = []
    scalers = {}
    pcas = {}
    for field, cols in cols_by_field.items():
        Xf = emb_df[cols].values
        if scale:
            sc = StandardScaler()
            Xf = sc.fit_transform(Xf)
            scalers[field] = sc
        # choose n_components: if provided, use it; else use min(128, dim)
        default_n = min(128, Xf.shape[1])
        ncomp = n_components_per_col.get(field, default_n) if n_components_per_col else default_n
        ncomp = min(ncomp, Xf.shape[1])
        pca = PCA(n_components=ncomp, svd_solver='randomized', random_state=RANDOM_STATE)
        Xf_red = pca.fit_transform(Xf)
        pcas[field] = pca
        cols_red = [f"pca__{field}__{i}" for i in range(Xf_red.shape[1])]
        out_frames.append(pd.DataFrame(Xf_red, columns=cols_red, index=emb_df.index))
    X_final = pd.concat(out_frames, axis=1)
    return X_final, scalers, pcas

# -------------------------
# Optuna objective: tune per-column n_components (coarse) and classifier params
# -------------------------
def optuna_search(X_emb_scaled: np.ndarray, y: np.ndarray, groups: np.ndarray, fields: List[str], max_pc_options=[64,96,128,192], n_trials=OPTUNA_TRIALS):
    cv = GroupKFold(n_splits=CV_SPLITS)
    sampler = TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)

    def objective(trial):
        # per-field n_components selection (categorical over small set)
        n_components_map = {}
        for f in fields:
            n_components_map[f] = trial.suggest_categorical(f"ncomp_{f}", max_pc_options)

        # Build reduced matrix on the fly by slicing precomputed PCA projections
        # For speed we will compute PCA per field once per trial here (this is OK for modest n_trials)
        # But in our outer code we called build_embedding_features with default dims; here we mimic per-trial PCA:
        X_blocks = []
        start = 0
        # The input X_emb_scaled contains concatenated original embeddings per field in order fields.
        # We'll split by original embedding dims per field — to avoid complexity, we will pass
        # in emb_df (DataFrame) and compute PCA per field inside objective. So change signature:
        raise RuntimeError("This helper should be called from wrapper that passes emb_df (DataFrame) rather than preconcatenated array.")

    # We won't run here; caller will use a different objective that accepts emb_df directly.
    return study

# -------------------------
# Simpler: Optuna objective that accepts emb_df and does per-field PCA inside
# -------------------------
def optuna_search_embdf(emb_df: pd.DataFrame, y: np.ndarray, groups: np.ndarray, fields: List[str], max_pc_options=[64,96,128,192], n_trials=OPTUNA_TRIALS):
    cv = GroupKFold(n_splits=CV_SPLITS)
    sampler = TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)

    def objective(trial):
        # choose n_components per field
        n_components_map = {f: int(trial.suggest_categorical(f"ncomp__{f}", max_pc_options)) for f in fields}

        # build features for this trial
        frames = []
        for f in fields:
            cols = [c for c in emb_df.columns if f"emb__{f}__" in c]
            Xf = emb_df[cols].values
            sc = StandardScaler()
            Xf = sc.fit_transform(Xf)
            ncomp = min(n_components_map[f], Xf.shape[1])
            pca = PCA(n_components=ncomp, svd_solver='randomized', random_state=RANDOM_STATE)
            Xf_red = pca.fit_transform(Xf)
            frames.append(Xf_red)
        X_trial = np.concatenate(frames, axis=1)

        # classifier family
        clf_choice = trial.suggest_categorical("clf", ["lgbm", "rf", "logreg"])

        if clf_choice == "lgbm":
            try:
                import lightgbm as lgb
            except Exception:
                clf_choice = "rf"
            else:
                clf = lgb.LGBMClassifier(
                    num_leaves = trial.suggest_int("lgb_num_leaves", 31, 127),
                    max_depth = trial.suggest_int("lgb_max_depth", 5, 30),
                    learning_rate = trial.suggest_float("lgb_lr", 1e-3, 1e-1, log=True),
                    n_estimators = trial.suggest_int("lgb_n_estimators", 200, 1000, step=100),
                    random_state = RANDOM_STATE,
                    n_jobs = -1,
                    class_weight = "balanced"
                )
        if clf_choice == "rf":
            clf = RandomForestClassifier(
                n_estimators = trial.suggest_int("rf_n_estimators", 200, 800, step=100),
                max_depth = trial.suggest_int("rf_max_depth", 10, 80),
                min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 8),
                max_features = trial.suggest_categorical("rf_max_features", ["sqrt", "log2", 0.5]),
                random_state = RANDOM_STATE,
                n_jobs = -1,
                class_weight = "balanced",
                oob_score = True
            )
        if clf_choice == "logreg":
            clf = LogisticRegression(max_iter=5000, C=trial.suggest_float("lr_C", 1e-3, 10.0, log=True),
                                     solver="saga", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)

        # evaluate
        scores = cross_val_score(clf, X_trial, y, cv=cv, groups=groups, scoring="accuracy", n_jobs=-1)
        return float(np.mean(scores))

    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study

# -------------------------
# ENTRYPOINT: entire flow
# -------------------------
def run_all(df: pd.DataFrame,
            id_col: str = "id",
            target_col: str = "target",
            text_cols: List[str] = TEXT_COLS,
            ord_cols: List[str] = ORD_COLS,
            cat_cols: List[str] = CAT_COLS,
            embed_model: str = EMBEDDING_MODEL,
            batch_size: int = BATCH_SIZE,
            optuna_trials: int = OPTUNA_TRIALS):
    # 1) split by id into train/test (grouped holdout)
    train_df, test_df = split_by_id(df, id_col=id_col, test_size=0.3, random_state=RANDOM_STATE)
    train_groups = train_df[id_col].values
    test_groups = test_df[id_col].values
    y_train = train_df[target_col].values
    y_test = test_df[target_col].values

    # 2) precompute embeddings on training set (and test set later for final eval)
    print("Precomputing train embeddings...")
    emb_train = precompute_embeddings(train_df, text_cols, model_name=embed_model, batch_size=batch_size)
    print("Precomputing test embeddings (for final eval)...")
    emb_test = precompute_embeddings(test_df, text_cols, model_name=embed_model, batch_size=batch_size)

    # 3) Optuna search (per-column PCA dims + classifier)
    print("Starting Optuna search (per-column PCA dims + classifier)...")
    study = optuna_search_embdf(emb_train, y_train, train_groups, fields=[c for c in text_cols], max_pc_options=[64,96,128,192], n_trials=optuna_trials)

    print("Best params:", study.best_trial.params)
    print("Best CV score:", study.best_value)

    # 4) Build final features with chosen n_components (per-field)
    chosen = study.best_trial.params
    ncomps_map = {}
    for f in text_cols:
        ncomps_map[f] = int(chosen.get(f"ncomp__{f}", 128))
    # Compute final reduced embeddings for train and test
    frames_tr = []
    frames_te = []
    pca_models = {}
    scaler_models = {}
    for f in text_cols:
        cols = [c for c in emb_train.columns if f"emb__{f}__" in c]
        Xf_tr = emb_train[cols].values
        Xf_te = emb_test[cols].values
        sc = StandardScaler()
        Xf_tr_s = sc.fit_transform(Xf_tr)
        Xf_te_s = sc.transform(Xf_te)
        ncomp = min(ncomps_map[f], Xf_tr_s.shape[1])
        pca = PCA(n_components=ncomp, svd_solver='randomized', random_state=RANDOM_STATE)
        Xf_tr_red = pca.fit_transform(Xf_tr_s)
        Xf_te_red = pca.transform(Xf_te_s)
        pca_models[f] = pca
        scaler_models[f] = sc
        frames_tr.append(pd.DataFrame(Xf_tr_red, columns=[f"pca__{f}__{i}" for i in range(Xf_tr_red.shape[1])], index=train_df.index))
        frames_te.append(pd.DataFrame(Xf_te_red, columns=[f"pca__{f}__{i}" for i in range(Xf_te_red.shape[1])], index=test_df.index))

    X_train_final = pd.concat(frames_tr, axis=1)
    X_test_final = pd.concat(frames_te, axis=1)

    # 5) Train final classifier with best params
    best = study.best_trial.params
    clf_choice = best["clf"]
    if clf_choice == "lgbm":
        import lightgbm as lgb
        final_clf = lgb.LGBMClassifier(
            num_leaves = best.get("lgb_num_leaves"),
            max_depth = best.get("lgb_max_depth"),
            learning_rate = best.get("lgb_lr"),
            n_estimators = best.get("lgb_n_estimators"),
            random_state = RANDOM_STATE,
            n_jobs = -1,
            class_weight = "balanced"
        )
    elif clf_choice == "rf":
        final_clf = RandomForestClassifier(
            n_estimators = best.get("rf_n_estimators"),
            max_depth = best.get("rf_max_depth"),
            min_samples_leaf = best.get("rf_min_samples_leaf"),
            max_features = best.get("rf_max_features"),
            random_state = RANDOM_STATE,
            n_jobs = -1,
            class_weight = "balanced",
            oob_score = True
        )
    else:
        final_clf = LogisticRegression(max_iter=5000, C=best.get("lr_C"), solver="saga", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)

    print("Fitting final classifier on full training set...")
    final_clf.fit(X_train_final, y_train)

    # 6) Evaluate on test set (grouped holdout)
    from sklearn.metrics import accuracy_score, classification_report
    y_pred_test = final_clf.predict(X_test_final)
    test_acc = accuracy_score(y_test, y_pred_test)
    print("Test accuracy:", test_acc)
    print("Classification report on test set:\n", classification_report(y_test, y_pred_test))

    # 7) Save artifacts
    artifacts = {
        "scalers": scaler_models,
        "pcas": pca_models,
        "clf": final_clf,
        "embed_model_name": embed_model,
        "text_cols": text_cols
    }
    joblib.dump(artifacts, "final_embeddings_pipeline_artifacts.joblib")
    print("Saved artifacts -> final_embeddings_pipeline_artifacts.joblib")

    return {
        "study": study,
        "final_clf": final_clf,
        "X_train_final": X_train_final,
        "X_test_final": X_test_final,
        "y_train": y_train,
        "y_test": y_test,
        "artifacts": artifacts
    }

run_all(df)


Precomputing train embeddings...
Encoding column 'best_tasks_free' (576 rows) with all-mpnet-base-v2 ...


Batches: 100%|██████████| 9/9 [01:08<00:00,  7.65s/it]


Encoding column 'subopt_tasks_free' (576 rows) with all-mpnet-base-v2 ...


Batches: 100%|██████████| 9/9 [01:34<00:00, 10.46s/it]


Encoding column 'verify_method_free' (576 rows) with all-mpnet-base-v2 ...


...

Best trial: 32. Best value: 0.626946:  98%|█████████▊| 59/60 [12:19<00:17, 18.00s/it]

[I 2025-11-20 00:53:40,928] Trial 58 finished with value: 0.5776878092667566 and parameters: {'ncomp__best_tasks_free': 96, 'ncomp__subopt_tasks_free': 64, 'ncomp__verify_method_free': 128, 'clf': 'lgbm', 'lgb_num_leaves': 90, 'lgb_max_depth': 11, 'lgb_lr': 0.005379508988961487, 'lgb_n_estimators': 800}. Best is trial 32 with value: 0.6269455690508323.


Best trial: 32. Best value: 0.626946: 100%|██████████| 60/60 [12:21<00:00, 12.36s/it]


[I 2025-11-20 00:53:42,867] Trial 59 finished with value: 0.5958164642375169 and parameters: {'ncomp__best_tasks_free': 128, 'ncomp__subopt_tasks_free': 64, 'ncomp__verify_method_free': 128, 'clf': 'logreg', 'lr_C': 0.0033182323534231843}. Best is trial 32 with value: 0.6269455690508323.
Best params: {'ncomp__best_tasks_free': 96, 'ncomp__subopt_tasks_free': 64, 'ncomp__verify_method_free': 128, 'clf': 'logreg', 'lr_C': 0.0010369349971150277}
Best CV score: 0.6269455690508323
Fitting final classifier on full training set...
Test accuracy: 0.6265060240963856
Classification report on test set:
               precision    recall  f1-score   support

     ChatGPT       0.68      0.73      0.71        83
      Claude       0.67      0.54      0.60        83
      Gemini       0.54      0.60      0.57        83

    accuracy                           0.63       249
   macro avg       0.63      0.63      0.63       249
weighted avg       0.63      0.63      0.63       249

Saved artifacts -> 

{'study': <optuna.study.study.Study at 0x1dc35069300>,
 'final_clf': LogisticRegression(C=0.0010369349971150277, class_weight='balanced',
                    max_iter=5000, n_jobs=-1, random_state=22, solver='saga'),
 'X_train_final':      pca__best_tasks_free__0  pca__best_tasks_free__1  \
 0                   5.566606                 4.004159   
 1                  14.713634               -10.625539   
 2                  13.404431                -3.259750   
 3                  -1.585059                 8.333453   
 4                  14.825644                -2.214353   
 ..                       ...                      ...   
 571                 7.801798                -1.696319   
 572               -11.984387                -3.920604   
 573                -6.938127                 9.908780   
 574                13.954392                -4.241294   
 575                 3.343416                 2.967397   
 
      pca__best_tasks_free__2  pca__best_tasks_free__3  \
 0        

In [5]:
# resume_from_artifacts_direct.py
import numpy as np
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.model_selection import GroupKFold, cross_val_score
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings("ignore")

# -------------------------
# USER CONFIG
# -------------------------
ARTIFACT_PATH = "final_embeddings_pipeline_artifacts.joblib"
CV_SPLITS = 5
OPTUNA_TRIALS = 60
RANDOM_STATE = 22
EMBEDDING_MODEL = "all-mpnet-base-v2"
BATCH_SIZE = 64
# -------------------------

# Load your previous artifacts (PCA/scalers, text_cols)
artifacts = joblib.load(ARTIFACT_PATH)
scaler_models = artifacts["scalers"]
pca_models = artifacts["pcas"]
text_cols = artifacts["text_cols"]
embed_model_name = artifacts.get("embed_model_name", EMBEDDING_MODEL)

# Load your raw data with text columns + target
df = pd.read_csv("training_data_clean.csv")  # replace with your CSV path
y = df["target"].values
groups = df["id"].values  # or np.arange(len(df)) if no grouping

# Precompute embeddings from artifact model
def precompute_embeddings(df, text_cols, model_name=embed_model_name, batch_size=BATCH_SIZE):
    model = SentenceTransformer(model_name)
    emb_frames = []
    for col in text_cols:
        texts = df[col].fillna("").astype(str).tolist()
        print(f"Encoding column '{col}' ...")
        emb = model.encode(texts, show_progress_bar=True, batch_size=batch_size)
        emb = np.asarray(emb)
        cols = [f"emb__{col}__{i}" for i in range(emb.shape[1])]
        emb_frames.append(pd.DataFrame(emb, columns=cols, index=df.index))
    return pd.concat(emb_frames, axis=1)

emb_df = precompute_embeddings(df, text_cols, model_name=embed_model_name, batch_size=BATCH_SIZE)

# Apply existing scalers + PCA
def transform_with_artifacts(emb_df, text_cols, scalers, pcas):
    frames = []
    for f in text_cols:
        cols = [c for c in emb_df.columns if f"emb__{f}__" in c]
        Xf = emb_df[cols].values
        sc = scalers[f]
        Xf_s = sc.transform(Xf)
        pca = pcas[f]
        Xf_red = pca.transform(Xf_s)
        frames.append(Xf_red)
    return np.concatenate(frames, axis=1)

X_final = transform_with_artifacts(emb_df, text_cols, scaler_models, pca_models)

# -------------------------
# Optuna objective: classifier only
# -------------------------
def optuna_objective(trial, X, y, groups):
    clf_choice = trial.suggest_categorical("clf", ["lgbm", "rf", "logreg"])

    if clf_choice == "lgbm":
        clf = lgb.LGBMClassifier(
            num_leaves=trial.suggest_int("lgb_num_leaves", 31, 127),
            max_depth=trial.suggest_int("lgb_max_depth", 5, 30),
            learning_rate=trial.suggest_float("lgb_lr", 1e-3, 1e-1, log=True),
            n_estimators=trial.suggest_int("lgb_n_estimators", 200, 1000, step=100),
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced"
        )
    elif clf_choice == "rf":
        clf = RandomForestClassifier(
            n_estimators=trial.suggest_int("rf_n_estimators", 200, 800, step=100),
            max_depth=trial.suggest_int("rf_max_depth", 10, 80),
            min_samples_leaf=trial.suggest_int("rf_min_samples_leaf", 1, 8),
            max_features=trial.suggest_categorical("rf_max_features", ["sqrt", "log2", 0.5]),
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
            oob_score=True
        )
    else:
        clf = LogisticRegression(max_iter=5000, C=trial.suggest_float("lr_C", 1e-3, 10.0, log=True),
                                 solver="saga", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)

    cv = GroupKFold(n_splits=CV_SPLITS)
    scores = cross_val_score(clf, X, y, cv=cv, groups=groups, scoring="accuracy", n_jobs=-1)
    return float(np.mean(scores))

# -------------------------
# Run Optuna search
# -------------------------
sampler = TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(lambda trial: optuna_objective(trial, X_final, y, groups), n_trials=OPTUNA_TRIALS, show_progress_bar=True)

print("Best params:", study.best_trial.params)
print("Best CV score:", study.best_value)

# Train final classifier
best = study.best_trial.params
clf_choice = best["clf"]
if clf_choice == "lgbm":
    final_clf = lgb.LGBMClassifier(
        num_leaves = best.get("lgb_num_leaves"),
        max_depth = best.get("lgb_max_depth"),
        learning_rate = best.get("lgb_lr"),
        n_estimators = best.get("lgb_n_estimators"),
        random_state = RANDOM_STATE,
        n_jobs = -1,
        class_weight = "balanced"
    )
elif clf_choice == "rf":
    final_clf = RandomForestClassifier(
        n_estimators = best.get("rf_n_estimators"),
        max_depth = best.get("rf_max_depth"),
        min_samples_leaf = best.get("rf_min_samples_leaf"),
        max_features = best.get("rf_max_features"),
        random_state = RANDOM_STATE,
        n_jobs = -1,
        class_weight = "balanced",
        oob_score = True
    )
else:
    final_clf = LogisticRegression(max_iter=5000, C=best.get("lr_C"), solver="saga",
                                   class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)

final_clf.fit(X_final, y)
joblib.dump({"clf": final_clf, "scalers": scaler_models, "pcas": pca_models, "text_cols": text_cols}, ARTIFACT_PATH)
print(f"Updated artifact saved -> {ARTIFACT_PATH}")


KeyError: 'target'

In [9]:
# resume_from_artifacts.py
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.model_selection import GroupKFold, cross_val_score
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings("ignore")

# -------------------------
# USER CONFIG
# -------------------------
ARTIFACT_PATH = "final_embeddings_pipeline_artifacts.joblib"  # previous artifact
EMB_TRAIN_PATH = "cached_train_embeddings.npy"  # optional, precomputed embeddings
EMB_TEST_PATH = "cached_test_embeddings.npy"
Y_TRAIN_PATH = "y_train.npy"
Y_TEST_PATH = "y_test.npy"
CV_SPLITS = 5
OPTUNA_TRIALS = 100  # increase to explore more
RANDOM_STATE = 22
# -------------------------

# Load precomputed embeddings and labels
emb_train = np.load(EMB_TRAIN_PATH)  # shape (n_samples, n_features)
emb_test = np.load(EMB_TEST_PATH)
y_train = np.load(Y_TRAIN_PATH)
y_test = np.load(Y_TEST_PATH)

# Load previous artifacts (PCA, scalers, etc.)
artifacts = joblib.load(ARTIFACT_PATH)
scaler_models = artifacts["scalers"]
pca_models = artifacts["pcas"]
text_cols = artifacts["text_cols"]

# Rebuild features using saved PCA/scalers
def transform_with_artifacts(emb_df, text_cols, scalers, pcas):
    frames = []
    for f in text_cols:
        cols = [c for c in emb_df.columns if f"emb__{f}__" in c] if isinstance(emb_df, pd.DataFrame) else None
        Xf = emb_df[:, sum([scalers[c].mean_.shape[0] for c in text_cols[:text_cols.index(f)]]):sum([scalers[c].mean_.shape[0] for c in text_cols[:text_cols.index(f)+1]])] if isinstance(emb_df, np.ndarray) else emb_df[cols].values
        sc = scalers[f]
        Xf_s = sc.transform(Xf)
        pca = pcas[f]
        Xf_red = pca.transform(Xf_s)
        frames.append(Xf_red)
    return np.concatenate(frames, axis=1)

X_train_final = transform_with_artifacts(emb_train, text_cols, scaler_models, pca_models)
X_test_final = transform_with_artifacts(emb_test, text_cols, scaler_models, pca_models)

# -------------------------
# Optuna objective: classifier + per-field PCA tuning if desired
# -------------------------
def optuna_objective(trial, X, y, groups):
    # Here you can tune classifier params only (PCA already applied)
    clf_choice = trial.suggest_categorical("clf", ["lgbm", "rf", "logreg"])

    if clf_choice == "lgbm":
        clf = lgb.LGBMClassifier(
            num_leaves=trial.suggest_int("lgb_num_leaves", 31, 127),
            max_depth=trial.suggest_int("lgb_max_depth", 5, 30),
            learning_rate=trial.suggest_float("lgb_lr", 1e-3, 1e-1, log=True),
            n_estimators=trial.suggest_int("lgb_n_estimators", 200, 1000, step=100),
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced"
        )
    elif clf_choice == "rf":
        clf = RandomForestClassifier(
            n_estimators=trial.suggest_int("rf_n_estimators", 200, 800, step=100),
            max_depth=trial.suggest_int("rf_max_depth", 10, 80),
            min_samples_leaf=trial.suggest_int("rf_min_samples_leaf", 1, 8),
            max_features=trial.suggest_categorical("rf_max_features", ["sqrt", "log2", 0.5]),
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
            oob_score=True
        )
    else:
        clf = LogisticRegression(max_iter=5000, C=trial.suggest_float("lr_C", 1e-3, 10.0, log=True),
                                 solver="saga", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)

    cv = GroupKFold(n_splits=CV_SPLITS)
    scores = cross_val_score(clf, X, y, cv=cv, groups=groups, scoring="accuracy", n_jobs=-1)
    return float(scores.mean())

# -------------------------
# Run Optuna
# -------------------------
def run_optuna(X_train, y_train, groups, n_trials=OPTUNA_TRIALS):
    sampler = TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(lambda trial: optuna_objective(trial, X_train, y_train, groups), n_trials=n_trials, show_progress_bar=True)
    print("Best params:", study.best_trial.params)
    print("Best CV score:", study.best_value)
    return study

# Assume `groups` are IDs repeated per sample (or just np.arange(len(y_train)) if unknown)
groups = np.arange(len(y_train))
study = run_optuna(X_train_final, y_train, groups)

# Train final classifier with best params
best = study.best_trial.params
clf_choice = best["clf"]
if clf_choice == "lgbm":
    final_clf = lgb.LGBMClassifier(
        num_leaves = best.get("lgb_num_leaves"),
        max_depth = best.get("lgb_max_depth"),
        learning_rate = best.get("lgb_lr"),
        n_estimators = best.get("lgb_n_estimators"),
        random_state = RANDOM_STATE,
        n_jobs = -1,
        class_weight = "balanced"
    )
elif clf_choice == "rf":
    final_clf = RandomForestClassifier(
        n_estimators = best.get("rf_n_estimators"),
        max_depth = best.get("rf_max_depth"),
        min_samples_leaf = best.get("rf_min_samples_leaf"),
        max_features = best.get("rf_max_features"),
        random_state = RANDOM_STATE,
        n_jobs = -1,
        class_weight = "balanced",
        oob_score = True
    )
else:
    final_clf = LogisticRegression(max_iter=5000, C=best.get("lr_C"), solver="saga",
                                   class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)

print("Fitting final classifier on full training set...")
final_clf.fit(X_train_final, y_train)

# Evaluate
from sklearn.metrics import accuracy_score, classification_report
y_pred_test = final_clf.predict(X_test_final)
print("Test accuracy:", accuracy_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

# Save final pipeline
joblib.dump({"clf": final_clf, "scalers": scaler_models, "pcas": pca_models, "text_cols": text_cols}, ARTIFACT_PATH)
print(f"Artifacts updated -> {ARTIFACT_PATH}")


FileNotFoundError: [Errno 2] No such file or directory: 'cached_train_embeddings.npy'

In [6]:
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from optuna.samplers import TPESampler
import optuna
cv_groups = GroupKFold(n_splits=5)

preprocessor = create_preprocessor()

# === STEP 1: Better TF-IDF search for RF
def objective_tfidf(trial):
    """Optimize TF-IDF for Random Forest specifically"""
    
    # RF works better with MORE features (unlike linear models)
    max_features = trial.suggest_categorical('max_features', [1000, 2000, 3000, 5000, None])
    ngram_range = trial.suggest_categorical('ngram_range', [(1, 1), (1, 2), (1, 3)])
    min_df = trial.suggest_int('min_df', 1, 3)  # Lower - RF can handle noise
    max_df = trial.suggest_float('max_df', 0.8, 0.98)  # Higher - keep more features
    sublinear_tf = trial.suggest_categorical('sublinear_tf', [True, False])
    # RF doesn't need normalized features - try without norm
    norm = trial.suggest_categorical('norm', ['l2', None])
    use_idf = trial.suggest_categorical('use_idf', [True, False])
    
    text_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        ('tfidf', TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            sublinear_tf=sublinear_tf,
            norm=norm,
            use_idf=use_idf,
            stop_words='english'
        ))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('text', text_pipeline, text_cols),
            ('ord', preprocess_ord(), ord_cols),
            ('cat', preprocess_cat(), cat_cols),
        ]
    )
    
    # Use lighter RF for TF-IDF search
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=100,  # Light but enough to be representative
            max_depth=20,
            min_samples_leaf=5,
            class_weight='balanced',
            random_state=22,
            n_jobs=-1
        ))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=1
    )
    
    return scores.mean()

studytfidf = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22, n_startup_trials=20)  # More random exploration
)
studytfidf.optimize(objective_tfidf, n_trials=75, show_progress_bar=True)

tfidf_params = {
    "max_features": studytfidf.best_params.get("max_features"),
    "ngram_range": studytfidf.best_params.get("ngram_range", (1, 1)),
    "min_df": studytfidf.best_params.get("min_df", 2),
    "max_df": studytfidf.best_params.get("max_df", 0.95),
    "sublinear_tf": studytfidf.best_params.get("sublinear_tf", True),
    "norm": studytfidf.best_params.get("norm", "l2"),
    # if you also searched use_idf earlier: include it
    # "use_idf": studytfidf.best_params.get("use_idf", True)
}

# text pipeline using the winning TF-IDF params
best_text_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="")),
    ("combiner", MakeCorpus()),  # your transformer that combines text columns
    ("tfidf", TfidfVectorizer(
        max_features=tfidf_params["max_features"],
        ngram_range=tfidf_params["ngram_range"],
        min_df=tfidf_params["min_df"],
        max_df=tfidf_params["max_df"],
        sublinear_tf=tfidf_params["sublinear_tf"],
        norm=tfidf_params["norm"],
        stop_words="english"
    ))
])

# Build ColumnTransformer (unfitted)
best_preprocessor = ColumnTransformer(
    transformers=[
        ("text", best_text_pipeline, list(text_cols)),   # must be list or array-like of column names/indices
        ("ord", preprocess_ord(), list(ord_cols)),
        ("cat", preprocess_cat(), list(cat_cols)),
    ],
    remainder="drop",  # or "passthrough" if you want other columns kept
    sparse_threshold=0.3  # keep as sparse if many sparse features
)

# sanity print
print("Built best_preprocessor with TF-IDF params:", tfidf_params)


# === STEP 2: Aggressive RF tuning
def objective_rf(trial):
    """Optimize RF - focus on preventing overfit"""
    
    n_estimators = trial.suggest_int('n_estimators', 300, 1000, step=100)
    
    # Key overfit controls
    max_depth = trial.suggest_int('max_depth', 8, 25)  # Shallow trees generalize better
    min_samples_split = trial.suggest_int('min_samples_split', 20, 100)  # High split threshold
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 10, 50)  # High leaf threshold
    
    # Feature sampling - critical for high-dim text
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])  # Conservative
    
    # Bootstrap sampling - underused regularization
    max_samples = trial.suggest_float('max_samples', 0.4, 0.8)  # Subsample aggressively
    bootstrap = True  # Always use bootstrap with max_samples
    
    # Try different split criteria
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            max_samples=max_samples,
            bootstrap=bootstrap,
            criterion=criterion,
            class_weight='balanced',
            random_state=22,
            n_jobs=-1,
            oob_score=True  # Track out-of-bag score
        ))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=1
    )
    
    return scores.mean()

study_rf = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22, n_startup_trials=20)
)
study_rf.optimize(objective_rf, n_trials=100, show_progress_bar=True)

rf_best = study_rf.best_params.copy()

# Create the pipeline (use best_preprocessor built earlier)
best_rf_pipeline = Pipeline([
    ("preprocessor", best_preprocessor),   # MUST be unfitted ColumnTransformer
    ("clf", RandomForestClassifier(
        n_estimators = rf_best["n_estimators"],
        max_depth = rf_best["max_depth"],
        min_samples_split = rf_best["min_samples_split"],
        min_samples_leaf = rf_best["min_samples_leaf"],
        max_features = rf_best["max_features"],
        max_samples = rf_best.get("max_samples", None),
        bootstrap = rf_best.get("bootstrap", True),
        criterion = rf_best.get("criterion", "gini"),
        class_weight = "balanced",
        random_state = 22,
        n_jobs = -1,
        oob_score = True   # you used oob in tuning; keep it for diagnostics
    ))
])
# Check OOB score vs CV score
best_rf_pipeline.fit(x_train, y_train)
print(f"\nOOB Score: {best_rf_pipeline.named_steps['clf'].oob_score_:.3f}")
print(f"CV Score: {study_rf.best_value:.3f}")
print(f"Train Score: {best_rf_pipeline.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_rf_pipeline.score(x_train, y_train) - study_rf.best_value:.3f}")

c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-11-19 23:14:06,543] A new study created in memory with name: no-name-1a72445e-035d-4e00-8065-4a5d7077994c
  0%|          | 0/75 [00:00<?, ?it/s]c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  warnings.warn(message)
c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  war

[I 2025-11-19 23:14:09,501] Trial 0 finished with value: 0.6578947368421053 and parameters: {'max_features': 5000, 'ngram_range': (1, 3), 'min_df': 1, 'max_df': 0.9461511656969636, 'sublinear_tf': False, 'norm': 'l2', 'use_idf': True}. Best is trial 0 with value: 0.6578947368421053.


Best trial: 1. Best value: 0.672065:   3%|▎         | 2/75 [00:05<03:00,  2.48s/it]c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  warnings.warn(message)
c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  warnings.warn(message)
c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  warnings.warn(message)


[I 2025-11-19 23:14:11,631] Trial 1 finished with value: 0.6720647773279353 and parameters: {'max_features': 2000, 'ngram_range': (1, 1), 'min_df': 2, 'max_df': 0.9051721357682411, 'sublinear_tf': True, 'norm': None, 'use_idf': False}. Best is trial 1 with value: 0.6720647773279353.


Best trial: 1. Best value: 0.672065:   4%|▍         | 3/75 [00:08<03:17,  2.74s/it]c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  warnings.warn(message)
c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  warnings.warn(message)
c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 3) which is of type tuple.
  warnings.warn(message)


...

Best trial: 39. Best value: 0.677148:  95%|█████████▌| 95/100 [24:56<01:16, 15.29s/it]

[I 2025-11-19 23:44:02,091] Trial 94 finished with value: 0.6703103913630228 and parameters: {'n_estimators': 900, 'max_depth': 14, 'min_samples_split': 25, 'min_samples_leaf': 21, 'max_features': 'sqrt', 'max_samples': 0.5184188532776888, 'criterion': 'gini'}. Best is trial 39 with value: 0.6771479982006299.


Best trial: 39. Best value: 0.677148:  95%|█████████▌| 95/100 [25:00<01:18, 15.79s/it]


[W 2025-11-19 23:44:05,371] Trial 95 failed with parameters: {'n_estimators': 600, 'max_depth': 12, 'min_samples_split': 22, 'min_samples_leaf': 15, 'max_features': 'sqrt', 'max_samples': 0.5362393850349826, 'criterion': 'gini'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_26040\2762391692.py", line 155, in objective_rf
    scores = cross_val_score(
  File "c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\sklearn\model_selection\_validation.py", line 677, in cross_val_score
    cv_results = cross_validate(
  File "c:\Users\carol\Desktop\Chara

KeyboardInterrupt: 

In [18]:
# ALL MODELS COMBINED, GRIDSEARCH

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
import numpy as np
from xgboost import XGBClassifier

# ============================================
# SETUP: GroupKFold cross-validation
# ============================================
cv_groups = GroupKFold(n_splits=5)

# Get the best preprocessor from your earlier grid search
# (assuming you've already run the initial grid_search)
best_preprocessor = grid_search.best_estimator_.named_steps['preprocessor']

print("="*60)
print("TESTING MODELS WITH GROUPKFOLD")
print("="*60)

# ============================================
# MODEL 1: LOGISTIC REGRESSION
# ============================================
print("\n### 1. LOGISTIC REGRESSION ###")

pipeline_logreg = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

param_grid_logreg = {
    'classifier__C': [0.001, 0.01, 0.1, 1.0, 10.0]
}

grid_logreg = GridSearchCV(
    pipeline_logreg,
    param_grid_logreg,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_logreg.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_logreg.best_params_}")
print(f"Best VALIDATION score: {grid_logreg.best_score_:.3f}")
print(f"Training score: {grid_logreg.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_logreg.score(x_train, y_train) - grid_logreg.best_score_:.3f}")

# ============================================
# MODEL 2: LINEAR SVC
# ============================================
print("\n### 2. LINEAR SVC ###")

pipeline_svc = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LinearSVC(dual=True, max_iter=10000, class_weight='balanced'))
])

param_grid_svc = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0]
}

grid_svc = GridSearchCV(
    pipeline_svc,
    param_grid_svc,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_svc.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_svc.best_params_}")
print(f"Best VALIDATION score: {grid_svc.best_score_:.3f}")
print(f"Training score: {grid_svc.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_svc.score(x_train, y_train) - grid_svc.best_score_:.3f}")

# ============================================
# MODEL 3: NAIVE BAYES
# ============================================
print("\n### 3. NAIVE BAYES ###")

pipeline_nb = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', MultinomialNB())
])

param_grid_nb = {
    'classifier__alpha': [0.1, 0.5, 1.0, 2.0, 5.0]
}

grid_nb = GridSearchCV(
    pipeline_nb,
    param_grid_nb,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_nb.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_nb.best_params_}")
print(f"Best VALIDATION score: {grid_nb.best_score_:.3f}")
print(f"Training score: {grid_nb.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_nb.score(x_train, y_train) - grid_nb.best_score_:.3f}")

# ============================================
# MODEL 4: RANDOM FOREST
# ============================================
print("\n### 4. RANDOM FOREST ###")

pipeline_rf = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=22))
])

param_grid_rf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, 20],
    'classifier__min_samples_leaf': [5, 10, 20],
    'classifier__max_features': ['sqrt', 'log2']
}

grid_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_rf.best_params_}")
print(f"Best VALIDATION score: {grid_rf.best_score_:.3f}")
print(f"Training score: {grid_rf.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_rf.score(x_train, y_train) - grid_rf.best_score_:.3f}")

# ============================================
# MODEL 5: XGBOOST
# ============================================
from sklearn.preprocessing import LabelEncoder

# Encode the labels BEFORE splitting
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['target'])

# Now split with encoded target
def split_data(df):
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(
        students, test_size=0.3, random_state=22
    )

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]

    train_groups = train_df['id'].values
    test_groups = test_df['id'].values

    x_train = train_df.drop(columns=['target', 'target_encoded', 'id'])
    y_train = train_df['target_encoded']  # Use encoded version

    x_test = test_df.drop(columns=['target', 'target_encoded', 'id'])
    y_test = test_df['target_encoded']  # Use encoded version

    return x_train, x_test, y_train, y_test, train_groups, test_groups

# Split the data
x_train, x_test, y_train, y_test, train_groups, test_groups = split_data(df)

print("\n### 5. XGBOOST ###")
pipeline_xgb = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=22
    ))
])

param_grid_xgb = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3,5, 7],
    'classifier__learning_rate': [0.01,  0.1, 0.3],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0],
    'classifier__min_child_weight': [1, 3, 5]
}

grid_xgb = GridSearchCV(
    pipeline_xgb,
    param_grid_xgb,
    cv=cv_groups,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)


grid_xgb.fit(x_train, y_train, groups=train_groups)

print(f"Best params: {grid_xgb.best_params_}")
print(f"Best VALIDATION score: {grid_xgb.best_score_:.3f}")
print(f"Training score: {grid_xgb.score(x_train, y_train):.3f}")
print(f"Overfit gap: {grid_xgb.score(x_train, y_train) - grid_xgb.best_score_:.3f}")




TESTING MODELS WITH GROUPKFOLD

### 1. LOGISTIC REGRESSION ###
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best params: {'classifier__C': 0.1}
Best VALIDATION score: 0.667
Training score: 0.707
Overfit gap: 0.040

### 2. LINEAR SVC ###
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best params: {'classifier__C': 0.1}
Best VALIDATION score: 0.674
Training score: 0.804
Overfit gap: 0.130

### 3. NAIVE BAYES ###
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best params: {'classifier__alpha': 0.5}
Best VALIDATION score: 0.656
Training score: 0.757
Overfit gap: 0.100

### 4. RANDOM FOREST ###
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'classifier__max_depth': 5, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 100}
Best VALIDATION score: 0.677
Training score: 0.773
Overfit gap: 0.095

### 5. XGBOOST ###
Fitting 5 folds for each of 324 candidates, totalling 1620 fits


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\xgboost\training.py:199: UserWarning: [11:20:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best params: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__min_child_weight': 1, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}
Best VALIDATION score: 0.675
Training score: 0.990
Overfit gap: 0.314


In [19]:
# ALL MODELS COMBINED, OPTUNA OPTIMIZATION

from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import numpy as np
import optuna
from optuna.samplers import TPESampler

# ============================================
# SETUP: GroupKFold cross-validation
# ============================================
cv_groups = GroupKFold(n_splits=5)

# Get the best preprocessor from your earlier grid search
# (assuming you've already run the initial grid_search)
best_preprocessor = grid_search.best_estimator_.named_steps['preprocessor']

print("="*60)
print("TESTING MODELS WITH OPTUNA OPTIMIZATION")
print("="*60)

# ============================================
# MODEL 1: LOGISTIC REGRESSION
# ============================================
print("\n### 1. LOGISTIC REGRESSION ###")

def objective_logreg(trial):
    """Objective function for Logistic Regression"""
    C = trial.suggest_float('C', 0.001, 10.0, log=True)
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', LogisticRegression(
            C=C,
            max_iter=1000,
            class_weight='balanced',
            random_state=22
        ))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=-1
    )
    
    return scores.mean()

study_logreg = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22)
)
study_logreg.optimize(objective_logreg, n_trials=50, show_progress_bar=True)

# Train final model with best params
best_logreg = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LogisticRegression(
        C=study_logreg.best_params['C'],
        max_iter=1000,
        class_weight='balanced',
        random_state=22
    ))
])
best_logreg.fit(x_train, y_train)

print(f"Best params: {study_logreg.best_params}")
print(f"Best VALIDATION score: {study_logreg.best_value:.3f}")
print(f"Training score: {best_logreg.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_logreg.score(x_train, y_train) - study_logreg.best_value:.3f}")

# ============================================
# MODEL 2: LINEAR SVC
# ============================================
print("\n### 2. LINEAR SVC ###")

def objective_svc(trial):
    """Objective function for Linear SVC"""
    C = trial.suggest_float('C', 0.01, 10.0, log=True)
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', LinearSVC(
            C=C,
            dual=True,
            max_iter=10000,
            class_weight='balanced',
            random_state=22
        ))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=-1
    )
    
    return scores.mean()

study_svc = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22)
)
study_svc.optimize(objective_svc, n_trials=50, show_progress_bar=True)

# Train final model with best params
best_svc = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LinearSVC(
        C=study_svc.best_params['C'],
        dual=True,
        max_iter=10000,
        class_weight='balanced',
        random_state=22
    ))
])
best_svc.fit(x_train, y_train)

print(f"Best params: {study_svc.best_params}")
print(f"Best VALIDATION score: {study_svc.best_value:.3f}")
print(f"Training score: {best_svc.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_svc.score(x_train, y_train) - study_svc.best_value:.3f}")

# ============================================
# MODEL 3: NAIVE BAYES
# ============================================
print("\n### 3. NAIVE BAYES ###")

def objective_nb(trial):
    """Objective function for Naive Bayes"""
    alpha = trial.suggest_float('alpha', 0.1, 5.0)
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', MultinomialNB(alpha=alpha))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=-1
    )
    
    return scores.mean()

study_nb = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22)
)
study_nb.optimize(objective_nb, n_trials=50, show_progress_bar=True)

# Train final model with best params
best_nb = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', MultinomialNB(alpha=study_nb.best_params['alpha']))
])
best_nb.fit(x_train, y_train)

print(f"Best params: {study_nb.best_params}")
print(f"Best VALIDATION score: {study_nb.best_value:.3f}")
print(f"Training score: {best_nb.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_nb.score(x_train, y_train) - study_nb.best_value:.3f}")

# ============================================
# MODEL 4: RANDOM FOREST
# ============================================
print("\n### 4. RANDOM FOREST ###")

def objective_rf(trial):
    """Objective function for Random Forest"""
    n_estimators = trial.suggest_int('n_estimators', 100, 300)
    max_depth = trial.suggest_int('max_depth', 5, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    
    pipeline = Pipeline([
        ('preprocessor', best_preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            class_weight='balanced',
            random_state=22,
            n_jobs=1  # Set to 1 since we're using n_jobs=-1 in cross_val_score
        ))
    ])
    
    scores = cross_val_score(
        pipeline, x_train, y_train,
        cv=cv_groups.split(x_train, y_train, groups=train_groups),
        scoring='accuracy',
        n_jobs=-1
    )
    
    return scores.mean()

study_rf = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=22)
)
study_rf.optimize(objective_rf, n_trials=100, show_progress_bar=True)

# Train final model with best params
best_rf = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=study_rf.best_params['n_estimators'],
        max_depth=study_rf.best_params['max_depth'],
        min_samples_leaf=study_rf.best_params['min_samples_leaf'],
        max_features=study_rf.best_params['max_features'],
        class_weight='balanced',
        random_state=22
    ))
])
best_rf.fit(x_train, y_train)

print(f"Best params: {study_rf.best_params}")
print(f"Best VALIDATION score: {study_rf.best_value:.3f}")
print(f"Training score: {best_rf.score(x_train, y_train):.3f}")
print(f"Overfit gap: {best_rf.score(x_train, y_train) - study_rf.best_value:.3f}")

# ============================================
# SUMMARY: COMPARE ALL MODELS
# ============================================
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

models_summary = {
    'Logistic Regression': (study_logreg.best_value, best_logreg.score(x_train, y_train)),
    'Linear SVC': (study_svc.best_value, best_svc.score(x_train, y_train)),
    'Naive Bayes': (study_nb.best_value, best_nb.score(x_train, y_train)),
    'Random Forest': (study_rf.best_value, best_rf.score(x_train, y_train))
}

for model_name, (val_score, train_score) in models_summary.items():
    print(f"\n{model_name}:")
    print(f"  Validation: {val_score:.3f}")
    print(f"  Training: {train_score:.3f}")
    print(f"  Overfit gap: {train_score - val_score:.3f}")

# Find best model
best_model_name = max(models_summary.items(), key=lambda x: x[1][0])[0]
print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*60}")

# ============================================
# OPTIONAL: VISUALIZE OPTIMIZATION HISTORY
# ============================================
print("\n### OPTIONAL: Visualization Code ###")
print("# Uncomment to visualize optimization history:")
print("""
import optuna.visualization as vis
import matplotlib.pyplot as plt

# Optimization history for Random Forest (best model)
fig = vis.plot_optimization_history(study_rf)
fig.show()

# Parameter importance for Random Forest
fig = vis.plot_param_importances(study_rf)
fig.show()

# Parallel coordinate plot for Random Forest
fig = vis.plot_parallel_coordinate(study_rf)
fig.show()
""")


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-11-19 11:36:19,007] A new study created in memory with name: no-name-b19c5cde-c973-4cfc-bb70-a0e53e35d5a9


TESTING MODELS WITH OPTUNA OPTIMIZATION

### 1. LOGISTIC REGRESSION ###


Best trial: 0. Best value: 0.635717:   2%|▏         | 1/50 [00:08<06:42,  8.21s/it]

[I 2025-11-19 11:36:27,211] Trial 0 finished with value: 0.6357174988753936 and parameters: {'C': 0.006820907334120959}. Best is trial 0 with value: 0.6357174988753936.


Best trial: 1. Best value: 0.667072:   4%|▍         | 2/50 [00:11<04:20,  5.42s/it]

...

[I 2025-11-19 11:38:49,090] Trial 97 finished with value: 0.6842555105713 and parameters: {'n_estimators': 198, 'max_depth': 16, 'min_samples_leaf': 9, 'max_features': 'sqrt'}. Best is trial 96 with value: 0.6842555105713.


Best trial: 96. Best value: 0.684256:  99%|█████████▉| 99/100 [01:30<00:00,  1.09it/s]

[I 2025-11-19 11:38:49,988] Trial 98 finished with value: 0.6548358074673865 and parameters: {'n_estimators': 197, 'max_depth': 17, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 96 with value: 0.6842555105713.


Best trial: 96. Best value: 0.684256: 100%|██████████| 100/100 [01:31<00:00,  1.10it/s]


[I 2025-11-19 11:38:50,890] Trial 99 finished with value: 0.6704903283850652 and parameters: {'n_estimators': 190, 'max_depth': 18, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 96 with value: 0.6842555105713.
Best params: {'n_estimators': 196, 'max_depth': 16, 'min_samples_leaf': 9, 'max_features': 'sqrt'}
Best VALIDATION score: 0.684
Training score: 0.774
Overfit gap: 0.090

MODEL COMPARISON SUMMARY

Logistic Regression:
  Validation: 0.669
  Training: 0.700
  Overfit gap: 0.031

Linear SVC:
  Validation: 0.679
  Training: 0.752
  Overfit gap: 0.073

Naive Bayes:
  Validation: 0.663
  Training: 0.762
  Overfit gap: 0.099

Random Forest:
  Validation: 0.684
  Training: 0.774
  Overfit gap: 0.090

BEST MODEL: Random Forest

### OPTIONAL: Visualization Code ###
# Uncomment to visualize optimization history:

import optuna.visualization as vis
import matplotlib.pyplot as plt

# Optimization history for Random Forest (best model)
fig = vis.plot_optimization_history(study_r

Logistic Regression below

In [46]:
from sklearn.model_selection import GroupKFold


best_tfidf = grid_search.best_estimator_

grid_log_reg = {'classifier__C': [0.001, 0.01, 0.1, 1.]}
cv1 = GroupKFold(n_splits=5)
grid_search_logreg = GridSearchCV(best_tfidf, grid_log_reg,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_logreg.fit(x_train, y_train, groups = train_group)


print(f"Best log reg model parameters: {grid_search_logreg.best_params_}")
print(f"Best log reg model: {grid_search_logreg.best_estimator_}")
print(f"Best log reg model score: {grid_search_logreg.best_score_}")
best_model = grid_search_logreg.best_estimator_
train_acc = best_model.score(x_train, y_train)
print(train_acc)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.1s
[CV] END ..................................classifier__C=0.1; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.2s
[CV] END ..................................classi

In [47]:
# Trying linear SVC
from sklearn.svm import LinearSVC

best_preprocessor = best_tfidf.named_steps['preprocessor']

full_pipeline2 = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LinearSVC(dual=True))
])

grid_lin_svc= {'classifier__C': [0.01, 0.1, 1.0, 10.0], 'classifier__max_iter': [5000, 10000, 20000]}
grid_search_lin_svc = GridSearchCV(full_pipeline2, grid_lin_svc,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_lin_svc.fit(x_train, y_train, groups =train_group)

print(f"Best log reg model parameters: {grid_search_lin_svc.best_params_}")
print(f"Best log reg model: {grid_search_lin_svc.best_estimator_}")
print(f"Best log reg model score: {grid_search_lin_svc.best_score_}")
best_model_svc = grid_search_lin_svc.best_estimator_
train_acc_svc = best_model_svc.score(x_train, y_train)
print(train_acc_svc)




Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.2s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=20000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=20000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__ma

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.8s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.8s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.8s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   0.5s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   0.6s


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.5s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.7s
Best log reg model parameters: {'classifier__C': 0.1, 'classifier__max_iter': 5000}
Best log reg model: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='',
                                                                                 strategy='constant')),
                                                                  ('combiner',
                                           

In [48]:
# Trying Naive Bayes

from sklearn.naive_bayes import MultinomialNB

full_pipeline3 = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', MultinomialNB())
])

grid_lin_nb=  {'classifier__alpha': [0.1, 0.5, 1.0, 2.0]}
grid_search_nb = GridSearchCV(full_pipeline3, grid_lin_nb,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_nb.fit(x_train, y_train, groups=train_group)

print(f"Best log reg model parameters: {grid_search_nb.best_params_}")
print(f"Best log reg model: {grid_search_nb.best_estimator_}")
print(f"Best log reg model score: {grid_search_nb.best_score_}")
best_model_nb = grid_search_nb.best_estimator_
train_acc_nb = grid_search_nb.score(x_train, y_train)
print(train_acc_nb)


Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=1.0; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier

In [49]:
# Trying random forests

from sklearn.ensemble import RandomForestClassifier
full_pipeline4 = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', RandomForestClassifier(random_state=22))
])

grid_rf = {
    'classifier__n_estimators': [100, 300],
    'classifier__max_depth': [None, 10, 20],
    'classifier__max_features': ['sqrt', 'log2']
}

grid_search_rf = GridSearchCV(full_pipeline4, grid_rf,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_rf.fit(x_train, y_train, groups=train_group)

print(f"Best log reg model parameters: {grid_search_rf.best_params_}")
print(f"Best log reg model: {grid_search_rf.best_estimator_}")
print(f"Best log reg model score: {grid_search_rf.best_score_}")
best_model_rf = grid_search_rf.best_estimator_
train_acc_rf = grid_search_rf.score(x_train, y_train)
print(train_acc_rf)


Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__n_estimators=100; total time=   0.4s
[CV] END classifier__max_depth=None, classifier__max_features=log2, classifier__n_estimators=100; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__max_features=log2, classifier__n_estimators=100; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__max_features=log2, classifier__n_estimators=100; total time=   0.2s
[CV

In [10]:
# TF=IDF DEVELOPMENT IN ISOLATION
"""CORPUS:
├── Document 1 (Row 1): col1 + col2 + col3 → Label: ChatGPT
├── Document 2 (Row 2): col1 + col2 + col3 → Label: Claude

Term frequency (TF): the raw count of a term in a document
Inverse document frequency (IDF): how important a word is, idf(t,D)=log(N/df(t))
"""
# GridSearchCV for hyperparam tuning
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: TAKE OUT WHEN YOU MOVE TO PIPELINE
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='constant', fill_value='')
df[text_cols] = imputer.fit_transform(df[text_cols])

# make corpus, create new column
df['corpus'] = df[text_cols].agg(' '.join, axis=1)
corpus = df['corpus']
labels = df['target']

# Create pipeline
pipeline = Pipeline([
    # TODO: also try removing stop words and adding max_df
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

# Define parameter grid
param_grid = {
    'tfidf__max_features': [500, 1000, 2000, 3000, None], # max number of features
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)], # length of context range
    'tfidf__min_df': [1, 2, 3, 4, 5] # minimum number times word has to appear to be included
}

# Search
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(corpus, labels)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy: {grid_search.best_score_:.3f}")




Best parameters: {'tfidf__max_features': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 3)}
Best accuracy: 0.605


WON'T BE RUNNING THIS BELOW

In [ ]:
# LLM EMBEDDINGS

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
# use pretrained weights

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def embed_texts(text_col, batch_size=32):
    """
    Accepts a pandas Series, list, numpy array, or single string and returns
    a numpy array of mean-pooled embeddings (N, hidden_size).
    Replaces missing values with empty strings so the tokenizer always gets str inputs.
    """
    # normalize input to a list of strings
    texts = [ '' if pd.isna(x) else str(x) for x in text_col ]

    if len(texts) == 0:
        return np.zeros((0, model.config.hidden_size))

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        # pass to model with no gradient bc we are not training, just getting embeddings
        with torch.no_grad():
            # done per response in batch  ============================================
            outputs = model(**enc) # outputs.last_hidden_state: (B, seq_len, 768)

            # Mean Pooling: average vectors of each true token vector in the text ====
            mask = enc["attention_mask"].unsqueeze(-1) # (B, seq_len, 1), add 1 dim for multiplication
            masked = outputs.last_hidden_state * mask # ignores padding tokens
            summed = masked.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts  # (B, hidden)

            all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# embed_best_tasks_free = embed_texts(df['best_tasks_free'])
# print(embed_best_tasks_free)
# # embed_subopt_tasks_free = embed_texts(df['subopt_tasks_free'])
# embed_verify_method_free = embed_texts(df['verify_method_free'])

def embed_texts(text_cols, df, batch_size=32):
    col_embedding = []
    for col in text_cols:
        embed_texts(df[col], batch_size=batch_size)
        col_embedding.append(col)
    return np.hstack(col_embedding)

print(embed_texts(text_cols, df))


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 